# Experiment 10: Reproduce Experiment 07 from Saved Sweep Artifacts

This notebook does no training or evaluation. It loads the per-run results produced by `runs/run_saes_sweep.py` and `runs/run_saes_sweep_eval.py`, then recreates the figures from notebook 07. The default is the `last` checkpoint because notebook 07 evaluated the final training step; set `VGSAE_CHECKPOINT_KIND=best` only for a separate best-training-loss analysis.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / 'outputs' / '.matplotlib'))

import matplotlib.pyplot as plt

from src.sae_sweep_plot import (
    load_sweep_plot_context, load_sweep_results, plot_data_overview,
    plot_mask_heatmaps,
    plot_reconstruction_metrics, plot_recovery_metrics,
    plot_sparsity_diagnostics, plot_support_metrics, plot_training_curves,
)

style_path = PROJECT_ROOT / 'physrev.mplstyle'
if style_path.exists():
    plt.style.use(style_path)

In [ ]:
SWEEP_DIR = Path(os.environ.get(
    'VGSAE_SWEEP_DIR',
    PROJECT_ROOT / 'outputs' / 'runs' / 'stage1_custom_baseline',
))
CHECKPOINT_KIND = os.environ.get('VGSAE_CHECKPOINT_KIND', 'last')
FIGURE_DIR = SWEEP_DIR / 'figures' / CHECKPOINT_KIND
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plot_context = load_sweep_plot_context(SWEEP_DIR)
final_df, history_df = load_sweep_results(SWEEP_DIR, CHECKPOINT_KIND)
print(f'{len(final_df)} evaluated runs from {SWEEP_DIR}')
print(plot_context)
final_df.head()

## Synthetic data overview

In [ ]:
plot_data_overview(SWEEP_DIR, FIGURE_DIR / 'data_overview.png');
plt.show()

## Reconstruction and recovery metrics

In [ ]:
plot_reconstruction_metrics(
    final_df, target_model_density=plot_context['target_model_density'],
    sae_width=plot_context['sae_width'],
    output_path=FIGURE_DIR / 'reconstruction_metrics.png',
);
plot_recovery_metrics(
    final_df, target_model_density=plot_context['target_model_density'],
    sae_width=plot_context['sae_width'],
    output_path=FIGURE_DIR / 'recovery_metrics.png',
);
plt.show()

## Support and sparsity diagnostics

In [ ]:
plot_support_metrics(
    final_df, target_model_density=plot_context['target_model_density'],
    sae_width=plot_context['sae_width'],
    output_path=FIGURE_DIR / 'support_metrics.png',
);
plot_sparsity_diagnostics(
    final_df, target_model_density=plot_context['target_model_density'],
    sae_width=plot_context['sae_width'],
    output_path=FIGURE_DIR / 'sparsity_diagnostics.png',
);
plt.show()

## Training curves and representative masks

In [ ]:
plot_training_curves(history_df, FIGURE_DIR / 'training_curves.png');
_, representatives = plot_mask_heatmaps(
    SWEEP_DIR, final_df,
    target_model_density=plot_context['target_model_density'],
    checkpoint_kind=CHECKPOINT_KIND,
    output_path=FIGURE_DIR / 'mask_heatmaps.png',
)
plt.show()
representatives[[
    'method_label', 'control_name', 'control_value', 'rho_model', 'selection_error'
]]